# Generar un dataset de recortes con SAM 3

Visión Artificial · CUTLAJO

Este notebook hace, paso a paso, lo mismo que `scripts/sam3_recortar_dataset_transformers.py`:

1. **Cargar** SAM 3 desde Hugging Face (`transformers`).
2. **Segmentar**: le damos una imagen y un prompt de texto ("round cell", "hard hat"...) y regresa *todas* las instancias con máscara, caja y score.
3. **Filtrar**: quitamos detecciones muy chicas, muy grandes o duplicadas.
4. **Recortar** cada objeto y guardarlo en una carpeta por clase.
5. **Revisar** con un mosaico y descargar el dataset como `.zip`.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → **T4 GPU**.

## 0. Revisar la GPU

In [ ]:
import torch

if torch.cuda.is_available():
    DISPOSITIVO = "cuda"
elif torch.backends.mps.is_available():   # Mac con Apple Silicon
    DISPOSITIVO = "mps"
else:
    DISPOSITIVO = "cpu"
print("Dispositivo:", DISPOSITIVO)
if DISPOSITIVO == "cpu":
    print("AVISO: sin GPU va a ser muy lento. En Colab activa la T4.")

## 1. Instalar dependencias

Colab ya trae `torch`, `numpy`, `pillow` y `matplotlib`. Falta `transformers` reciente (SAM 3 llegó en la versión 5).

In [ ]:
%pip install -q "transformers>=5" opencv-python

## 2. Iniciar sesión en Hugging Face

Los pesos de SAM 3 son de acceso restringido:

1. Pide acceso en https://huggingface.co/facebook/sam3 (tarda un rato en aprobarse).
2. Crea un token de tipo *Read* en https://huggingface.co/settings/tokens.
3. En Colab, guárdalo en el panel de la llave 🔑 (Secrets) con el nombre `HF_TOKEN` y activa el acceso al notebook. Si no, la celda te lo pide.

In [ ]:
import os
from huggingface_hub import login

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")

if token:
    login(token=token)
else:
    from huggingface_hub import notebook_login
    notebook_login()

## 3. Subir las imágenes

Dos opciones:

- **A.** Subirlas directo (se pierden al cerrar la sesión).
- **B.** Montar Google Drive y apuntar a una carpeta.

Fuera de Colab, solo cambia `DIR_ENTRADA` a tu carpeta.

In [ ]:
from pathlib import Path

DIR_ENTRADA = Path("raw")
DIR_ENTRADA.mkdir(exist_ok=True)

try:
    from google.colab import files
    subidos = files.upload()   # opción A: abre el selector de archivos
    for nombre, contenido in subidos.items():
        (DIR_ENTRADA / nombre).write_bytes(contenido)
except ImportError:
    print("No estás en Colab: copia tus imágenes a", DIR_ENTRADA.resolve())

# Opción B (Drive): descomenta estas líneas
# from google.colab import drive
# drive.mount("/content/drive")
# DIR_ENTRADA = Path("/content/drive/MyDrive/vision/raw")

EXTENSIONES = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
imagenes = sorted(p for p in DIR_ENTRADA.iterdir() if p.suffix.lower() in EXTENSIONES)
print(len(imagenes), "imágenes:", [p.name for p in imagenes[:10]])

## 4. Cargar SAM 3

La primera vez baja ~3.4 GB. `Sam3Processor` prepara la imagen y el texto; `Sam3Model` es la red.

In [ ]:
from transformers import Sam3Model, Sam3Processor

modelo = Sam3Model.from_pretrained("facebook/sam3").to(DISPOSITIVO).eval()
procesador = Sam3Processor.from_pretrained("facebook/sam3")
print("Parámetros: %.0f M" % (sum(p.numel() for p in modelo.parameters()) / 1e6))

## 5. Probar con una imagen

Aquí está el corazón del pipeline. Dos detalles importantes:

- El **encoder de imagen** (la parte cara) corre **una sola vez** por imagen con `get_vision_features`. Si tenemos varios prompts, todos reutilizan esas features.
- SAM 3 regresa **todas** las instancias que coinciden con el prompt, no solo una.

Los prompts van **en inglés**, como frases nominales cortas. Si no encuentra nada, prueba sinónimos más genéricos: con fotos de microscopio, "oocyte" no da nada pero "round cell" sí.

In [ ]:
import numpy as np
from PIL import Image, ImageOps


def abrir_imagen(ruta):
    """RGB respetando la orientación EXIF (fotos de celular)."""
    return ImageOps.exif_transpose(Image.open(ruta)).convert("RGB")


@torch.inference_mode()
def segmentar(imagen, frases, umbral=0.5):
    """Regresa {frase: {"masks", "boxes", "scores"}} para cada prompt."""
    entradas = procesador(images=imagen, return_tensors="pt").to(DISPOSITIVO)
    features = modelo.get_vision_features(pixel_values=entradas.pixel_values)   # una vez
    resultados = {}
    for frase in frases:
        texto = procesador(text=frase, return_tensors="pt").to(DISPOSITIVO)
        salida = modelo(vision_embeds=features, input_ids=texto.input_ids,
                        attention_mask=texto.attention_mask)
        r = procesador.post_process_instance_segmentation(
            salida, threshold=umbral, mask_threshold=0.5,
            target_sizes=entradas["original_sizes"].tolist())[0]
        resultados[frase] = {
            "masks": r["masks"].cpu().numpy().astype(bool),
            "boxes": r["boxes"].float().cpu().numpy().round().astype(int),
            "scores": r["scores"].float().cpu().numpy(),
        }
    return resultados


PROMPT_PRUEBA = "round cell"   # <- cámbialo por tu objeto
imagen = abrir_imagen(imagenes[0])
res = segmentar(imagen, [PROMPT_PRUEBA])[PROMPT_PRUEBA]
print(f"{len(res['scores'])} instancias, scores:", np.round(res["scores"], 2))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def dibujar(imagen, res, titulo=""):
    arr = np.asarray(imagen).astype(float) / 255
    rng = np.random.default_rng(0)
    for m in res["masks"]:
        color = rng.random(3)
        arr[m] = arr[m] * 0.5 + color * 0.5
    fig, ax = plt.subplots(figsize=(10, 7.5))
    ax.imshow(arr)
    for (x0, y0, x1, y1), s in zip(res["boxes"], res["scores"]):
        ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color="yellow", lw=1.5))
        ax.text(x0, y0 - 5, f"{s:.2f}", color="yellow", fontsize=9, weight="bold")
    ax.set_title(titulo)
    ax.axis("off")
    plt.show()


dibujar(imagen, res, f'"{PROMPT_PRUEBA}" en {imagenes[0].name}')

Fíjate en dos cosas típicas:

- **Objetos anidados.** En los óvulos sale una máscara para el citoplasma y otra para el óvulo completo con su corona. No son duplicados: se traslapan poco.
- **Basura chica.** Restos o grumos con score decente pero área diminuta. Para eso son los filtros del siguiente paso.

## 6. Filtros de calidad

| Parámetro | Qué hace |
|---|---|
| `UMBRAL` | confianza mínima de SAM 3 |
| `AREA_MIN` / `AREA_MAX` | fracción de la imagen que ocupa la máscara |
| `LADO_MIN` | ancho y alto mínimos de la caja, en px |
| `IOU_DUPLICADO` | si dos máscaras de la misma clase se traslapan más que esto, se queda la de mayor score |

In [ ]:
# Prompts en formato  clase -> frase en inglés.  La clase es el nombre de la carpeta.
PROMPTS = {"ovocito": "round cell"}

UMBRAL = 0.5
AREA_MIN = 0.01
AREA_MAX = 0.95
LADO_MIN = 32
IOU_DUPLICADO = 0.85
PADDING = 0.05          # margen extra alrededor de la caja
FONDO = "original"      # "original", "transparente", "negro" o "blanco"
TAMANO = 0              # 0 = tamaño original; 224 = cuadrado de 224 px


def iou(a, b):
    union = np.logical_or(a, b).sum()
    return np.logical_and(a, b).sum() / union if union else 0.0


def filtrar(dets, ancho, alto):
    buenas = []
    for d in dets:
        x0, y0, x1, y1 = d["caja"]
        d["area"] = d["mascara"].sum() / (ancho * alto)
        if not AREA_MIN <= d["area"] <= AREA_MAX:
            continue
        if (x1 - x0) < LADO_MIN or (y1 - y0) < LADO_MIN:
            continue
        buenas.append(d)
    buenas.sort(key=lambda d: d["score"], reverse=True)
    unicas = []
    for d in buenas:
        if all(u["clase"] != d["clase"] or iou(u["mascara"], d["mascara"]) < IOU_DUPLICADO for u in unicas):
            unicas.append(d)
    return unicas

## 7. Recortar y guardar

Para cada detección que pasa los filtros: recorta la caja con un poco de margen, opcionalmente borra el fondo usando la máscara, y guarda el archivo en `dataset/recortes/<clase>/`. También escribe `metadatos.csv` con el score, la caja y el área de cada recorte.

In [ ]:
import csv


def recortar(imagen, d):
    ancho, alto = imagen.size
    x0, y0, x1, y1 = d["caja"]
    pw, ph = (x1 - x0) * PADDING, (y1 - y0) * PADDING
    x0, y0 = max(0, int(x0 - pw)), max(0, int(y0 - ph))
    x1, y1 = min(ancho, int(x1 + pw)), min(alto, int(y1 + ph))
    region = imagen.crop((x0, y0, x1, y1))
    if FONDO != "original":
        alfa = Image.fromarray((d["mascara"][y0:y1, x0:x1] * 255).astype(np.uint8), mode="L")
        if FONDO == "transparente":
            region = region.convert("RGBA")
            region.putalpha(alfa)
        else:
            lienzo = Image.new("RGB", region.size, (0, 0, 0) if FONDO == "negro" else (255, 255, 255))
            lienzo.paste(region, (0, 0), alfa)
            region = lienzo
    if TAMANO:
        cuadro = Image.new(region.mode, (TAMANO, TAMANO))
        region.thumbnail((TAMANO, TAMANO), Image.Resampling.LANCZOS)
        cuadro.paste(region, ((TAMANO - region.width) // 2, (TAMANO - region.height) // 2))
        region = cuadro
    return region


DIR_SALIDA = Path("dataset")
ext = ".png" if FONDO == "transparente" else ".jpg"
guardados = {c: [] for c in PROMPTS}
for c in PROMPTS:
    (DIR_SALIDA / "recortes" / c).mkdir(parents=True, exist_ok=True)

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **k: x

with open(DIR_SALIDA / "metadatos.csv", "w", newline="") as f:
    csv_out = csv.writer(f)
    csv_out.writerow(["archivo", "clase", "origen", "score", "x0", "y0", "x1", "y1", "area"])
    for ruta in tqdm(imagenes, desc="Segmentando"):
        imagen = abrir_imagen(ruta)
        resultados = segmentar(imagen, list(PROMPTS.values()), UMBRAL)
        dets = []
        for clase, frase in PROMPTS.items():
            r = resultados[frase]
            for m, caja, s in zip(r["masks"], r["boxes"], r["scores"]):
                dets.append({"clase": clase, "mascara": m, "caja": tuple(caja), "score": float(s)})
        conteo = {}
        for d in filtrar(dets, *imagen.size):
            k = conteo[d["clase"]] = conteo.get(d["clase"], -1) + 1
            destino = DIR_SALIDA / "recortes" / d["clase"] / f"{ruta.stem}_{d['clase']}_{k:02d}{ext}"
            recorte = recortar(imagen, d)
            (recorte.convert("RGB") if ext == ".jpg" else recorte).save(destino)
            guardados[d["clase"]].append(destino)
            csv_out.writerow([destino.relative_to(DIR_SALIDA), d["clase"], ruta.name,
                              round(d["score"], 4), *d["caja"], round(float(d["area"]), 5)])

for c, rutas in guardados.items():
    print(f"{c:<15} {len(rutas)} recortes")

## 8. Revisar el mosaico

**Siempre** revisa antes de entrenar. SAM 3 etiqueta muy bien, pero no distingue variantes específicas de un objeto (un óvulo sano de uno degenerado, *tu* taza de cualquier taza). Si ves basura, ajusta `AREA_MIN` o `UMBRAL` en el paso 6 y vuelve a correr el 7.

In [ ]:
def mosaico(rutas, lado=128, columnas=10, maximo=100):
    rutas = rutas[:maximo]
    filas = (len(rutas) + columnas - 1) // columnas
    hoja = Image.new("RGB", (columnas * lado, max(1, filas) * lado), (40, 40, 40))
    for i, ruta in enumerate(rutas):
        im = Image.open(ruta).convert("RGB")
        im.thumbnail((lado, lado))
        hoja.paste(im, ((i % columnas) * lado + (lado - im.width) // 2,
                        (i // columnas) * lado + (lado - im.height) // 2))
    return hoja


for c, rutas in guardados.items():
    if rutas:
        hoja = mosaico(rutas)
        hoja.save(DIR_SALIDA / f"mosaico_{c}.jpg", quality=90)
        plt.figure(figsize=(12, 12 * hoja.height / hoja.width))
        plt.imshow(hoja)
        plt.title(c)
        plt.axis("off")
        plt.show()

## 9. Descargar el dataset

In [ ]:
import shutil

zip_ruta = shutil.make_archive("dataset_sam3", "zip", DIR_SALIDA)
print("Listo:", zip_ruta)
try:
    from google.colab import files
    files.download(zip_ruta)
except ImportError:
    pass

## Siguientes pasos

- El script `scripts/sam3_recortar_dataset_transformers.py` del repo hace todo esto desde la terminal y agrega: filtro por imágenes de referencia (DINOv2), exportación YOLO-seg para Roboflow/Ultralytics y `resumen.json` con los parámetros de la corrida.
- La versión `scripts/sam3_recortar_dataset_oficial.py` usa el código de Meta y el archivo `sam3.pt`. En Colab también corre (es Linux con CUDA).